# 2AFC Constant-Stimuli Psychophysics Analysis

This notebook analyzes a two-alternative forced-choice (2AFC) constant-stimuli experiment.

**Raw data rule:** set `DATA_ROOT` to the main data folder. The notebook treats it as read-only and writes all outputs to `OUTPUT_ROOT`.

**Important fitting rule:** the raw `answer` column is **not** used directly for fitting. Each trial is first converted into `response_comparison_greater` after determining whether the comparison stimulus was object 1 or object 2.

## 0. Configuration

Edit `DATA_ROOT`. Keep `OUTPUT_ROOT` outside the raw data folder.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# Works whether Jupyter's current directory is the project root, analysis/, or this folder.
CWD = Path.cwd()
if (CWD / "twoafc_psychophysics.py").exists():
    ANALYSIS_DIR = CWD
elif (CWD / "analysis" / "psychophysics_analysis" / "twoafc_psychophysics.py").exists():
    ANALYSIS_DIR = CWD / "analysis" / "psychophysics_analysis"
elif (CWD / "psychophysics_analysis" / "twoafc_psychophysics.py").exists():
    ANALYSIS_DIR = CWD / "psychophysics_analysis"
else:
    raise FileNotFoundError("Could not locate twoafc_psychophysics.py. Open the notebook from the project root, analysis/, or analysis/psychophysics_analysis/.")
sys.path.insert(0, str(ANALYSIS_DIR))
import twoafc_psychophysics as pf

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

# === EDIT THIS PATH "PUT_MAIN_DATA_FOLDER_HERE" ===
DATA_ROOT = Path(r"C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\Codes\Parallel_Heptics\debug_experiments")

# Separate output location. Do not place this inside DATA_ROOT.
OUTPUT_ROOT = ANALYSIS_DIR / "results"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

STANDARD_FALLBACK = 85.0          # used only as fallback/tie-break anchor
STANDARD_ABS_TOLERANCE = 0.75     # tolerance for classifying the inferred standard
FIG_DPI = 160

# Optional manual column overrides if automatic detection needs help.
# Example: {"object_1_value": "object_1_stiffness", "answer": "answer"}
MANUAL_COLUMN_OVERRIDES = {}

print("ANALYSIS_DIR:", ANALYSIS_DIR.resolve())
print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT.resolve())
if str(DATA_ROOT) == "PUT_MAIN_DATA_FOLDER_HERE":
    print("?? Set DATA_ROOT before running the analysis cells.")


ANALYSIS_DIR: C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\Codes\Parallel_Heptics\analysis\psychophysics_analysis
DATA_ROOT: PUT_MAIN_DATA_FOLDER_HERE
OUTPUT_ROOT: C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\Codes\Parallel_Heptics\analysis\psychophysics_analysis\results
?? Set DATA_ROOT before running the analysis cells.


## 1. File discovery and loading

One folder per subject is expected under `DATA_ROOT`. The code searches each subject folder recursively for files named exactly `answers.csv` only, then saves a discovery audit table.

In [ ]:
pf.validate_paths(DATA_ROOT, OUTPUT_ROOT)

file_discovery_summary = pf.discover_answer_files(DATA_ROOT, OUTPUT_ROOT)
pf.save_csv(file_discovery_summary, OUTPUT_ROOT, "file_discovery_summary.csv")
display(file_discovery_summary.head(30))

combined_raw_imported_data = pf.load_selected_subject_csvs(file_discovery_summary)
pf.save_csv(combined_raw_imported_data, OUTPUT_ROOT, "combined_raw_imported_data.csv")
display(combined_raw_imported_data.head())
print("combined raw shape:", combined_raw_imported_data.shape)

## 2. Column detection and validation

Detects object values, object fingers, answer, reaction time, trial index, timestamp, and optional block columns. Review the detection table; if needed, set `MANUAL_COLUMN_OVERRIDES` above and rerun.

In [ ]:
detected_columns, column_detection_details = pf.detect_columns(combined_raw_imported_data, MANUAL_COLUMN_OVERRIDES)
print(json.dumps(detected_columns, indent=2, ensure_ascii=False))
pf.save_csv(column_detection_details, OUTPUT_ROOT, "column_detection_details.csv")
display(column_detection_details.groupby("target").head(5))

missing_required = [k for k in ["object_1_value", "object_2_value", "answer"] if not detected_columns.get(k)]
if missing_required:
    raise RuntimeError(f"Missing required detected columns: {missing_required}. Add MANUAL_COLUMN_OVERRIDES and rerun.")

## 3. Trial cleaning and canonicalization

Protocol rows and invalid/ambiguous trials are flagged and saved separately. Valid trials receive `response_comparison_greater` for fitting.

Examples:
- standard object 1, comparison object 2 ? response is 1 when `answer == 1`.
- standard object 2, comparison object 1 ? response is 1 when `answer == 0`.

In [ ]:
inferred_standard_value, standard_inference_table = pf.infer_standard_value(combined_raw_imported_data, detected_columns, STANDARD_FALLBACK)
print(f"Inferred standard/reference value: {inferred_standard_value:g}")
pf.save_csv(standard_inference_table, OUTPUT_ROOT, "standard_value_inference.csv")
display(standard_inference_table.head(10))

combined_clean_trials, combined_flagged_trials = pf.canonicalize_trials(
    combined_raw_imported_data,
    detected_columns,
    standard_value=inferred_standard_value,
    standard_tolerance=STANDARD_ABS_TOLERANCE,
)
pf.save_csv(combined_clean_trials, OUTPUT_ROOT, "combined_clean_trials.csv")
pf.save_csv(combined_flagged_trials, OUTPUT_ROOT, "combined_flagged_trials.csv")

print("Clean trials:", combined_clean_trials.shape)
display(combined_clean_trials.head())
print("Flagged trials:", combined_flagged_trials.shape)
display(combined_flagged_trials.head())

## 4. Block and finger detection

Finger labels are normalized to `I`, `M`, `R`, `P`. Block order is inferred from changes in `finger_condition` within each subject.

In [ ]:
combined_clean_trials, block_order_summary = pf.add_block_numbers(combined_clean_trials)
pf.save_csv(combined_clean_trials, OUTPUT_ROOT, "combined_clean_trials.csv")
pf.save_csv(block_order_summary, OUTPUT_ROOT, "block_order_summary.csv")
display(block_order_summary.head(30))
display(pd.crosstab(combined_clean_trials["subject_id"], combined_clean_trials["finger_condition"]))

## 5. Quality-control summaries

Flags suspicious cases without silently deleting more data: missing/invalid rows, trial counts per value/finger, non-monotonic curves, apparent error rate, RT outliers, side bias, standard-side bias, and related warnings.

In [ ]:
qc_summary = pf.make_qc_summary(combined_clean_trials, combined_flagged_trials)
pf.save_csv(qc_summary, OUTPUT_ROOT, "qc_summary.csv")
display(qc_summary)

## 6. Psychometric aggregation

Aggregates valid trials into binomial counts: `comparison_value`, `n_trials`, `n_comparison_greater`, and observed probability.

In [ ]:
psychometric_input_by_subject_finger = pf.make_psychometric_input(combined_clean_trials, ["subject_id", "finger_condition"])
pf.save_csv(psychometric_input_by_subject_finger, OUTPUT_ROOT, "psychometric_input_by_subject_finger.csv")
display(psychometric_input_by_subject_finger.head(30))

## Psychometric fitting setup

The notebook attempts Wichmann-lab `psignifit` first when available and compatible. If not, it prints installation guidance and uses the scipy logistic fallback on physical stimulus values.

In [ ]:
PSIGNIFIT_AVAILABLE, PSIGNIFIT_STATUS = pf.check_psignifit_available()
print(PSIGNIFIT_STATUS)
if not PSIGNIFIT_AVAILABLE:
    print("Optional psignifit install example: pip install psignifit")
    print("Continuing with scipy logistic fallback fits.")

## 7. Subject ? finger psychometric fits

In [ ]:
pse_jnd_by_subject_finger = pf.fit_conditions(psychometric_input_by_subject_finger, ["subject_id", "finger_condition"], PSIGNIFIT_AVAILABLE)
pf.save_csv(pse_jnd_by_subject_finger, OUTPUT_ROOT, "pse_jnd_by_subject_finger.csv")
display(pse_jnd_by_subject_finger)

## 8. Subject-level pooled-across-fingers fits

In [ ]:
psychometric_input_subject_pooled = pf.make_psychometric_input(combined_clean_trials, ["subject_id"])
pse_jnd_subject_pooled = pf.fit_conditions(psychometric_input_subject_pooled, ["subject_id"], PSIGNIFIT_AVAILABLE)
pf.save_csv(psychometric_input_subject_pooled, OUTPUT_ROOT, "psychometric_input_subject_pooled.csv")
pf.save_csv(pse_jnd_subject_pooled, OUTPUT_ROOT, "pse_jnd_subject_pooled.csv")
display(pse_jnd_subject_pooled)

## 9. Group-level fits by finger

Pools all valid trials across subjects within each finger condition.

In [ ]:
psychometric_input_group_by_finger = pf.make_psychometric_input(combined_clean_trials, ["finger_condition"])
pse_jnd_group_by_finger = pf.fit_conditions(psychometric_input_group_by_finger, ["finger_condition"], PSIGNIFIT_AVAILABLE)
pf.save_csv(psychometric_input_group_by_finger, OUTPUT_ROOT, "psychometric_input_group_by_finger.csv")
pf.save_csv(pse_jnd_group_by_finger, OUTPUT_ROOT, "pse_jnd_group_by_finger.csv")
display(pse_jnd_group_by_finger)

## 10. Group-level all-pooled fit

In [ ]:
all_pooled_trials = combined_clean_trials.copy()
all_pooled_trials["group"] = "all_pooled"
psychometric_input_group_all_pooled = pf.make_psychometric_input(all_pooled_trials, ["group"])
pse_jnd_group_all_pooled = pf.fit_conditions(psychometric_input_group_all_pooled, ["group"], PSIGNIFIT_AVAILABLE)
pf.save_csv(psychometric_input_group_all_pooled, OUTPUT_ROOT, "psychometric_input_group_all_pooled.csv")
pf.save_csv(pse_jnd_group_all_pooled, OUTPUT_ROOT, "pse_jnd_group_all_pooled.csv")
display(pse_jnd_group_all_pooled)

## 11. Subject-average group analysis

Computes subject-level probabilities first, then averages subjects so each subject has equal weight. This complements trial-pooled group fits.

In [ ]:
subject_average_group_by_finger = pf.subject_average_psychometric(combined_clean_trials, ["finger_condition"])
subject_average_group_all = pf.subject_average_psychometric(combined_clean_trials.assign(group="all_subject_average"), ["group"])
pf.save_csv(subject_average_group_by_finger, OUTPUT_ROOT, "subject_average_group_by_finger.csv")
pf.save_csv(subject_average_group_all, OUTPUT_ROOT, "subject_average_group_all_pooled.csv")
display(subject_average_group_by_finger.head(30))

## 12. Order-effect analysis

Summarizes fatigue/order effects across trial number and inferred block order.

In [ ]:
order_effects_summary, order_effects_binned = pf.compute_order_effects(combined_clean_trials)
pf.save_csv(order_effects_summary, OUTPUT_ROOT, "order_effects_summary.csv")
pf.save_csv(order_effects_binned, OUTPUT_ROOT, "order_effects_binned.csv")
display(order_effects_summary)

## 13. Plots and saved outputs

Saves PNG figures: subject ? finger curves, group curves, all-pooled curve, PSE/JND summaries, order effects, side bias, and QC plots. Subject-specific outputs are also written under `OUTPUT_ROOT/subjects/<subject_id>/`.

In [ ]:
figure_paths = pf.save_all_figures(
    OUTPUT_ROOT,
    combined_clean_trials,
    qc_summary,
    psychometric_input_by_subject_finger,
    pse_jnd_by_subject_finger,
    psychometric_input_group_by_finger,
    pse_jnd_group_by_finger,
    psychometric_input_group_all_pooled,
    pse_jnd_group_all_pooled,
    order_effects_binned,
    FIG_DPI,
)
print(f"Saved {len(figure_paths)} figure files.")
display(pd.DataFrame({"figure": [str(p) for p in figure_paths]}).head(30))

pf.save_subject_level_outputs(
    OUTPUT_ROOT,
    combined_clean_trials,
    combined_flagged_trials,
    pse_jnd_by_subject_finger,
    pse_jnd_subject_pooled,
)
manifest = pf.analysis_manifest(OUTPUT_ROOT)
display(manifest)
print("Figures folder:", OUTPUT_ROOT / "figures")
print("Subject output folder:", OUTPUT_ROOT / "subjects")

## Interpretation notes

- `PSE` is the physical comparison value where `P(response_comparison_greater) = 0.5`.
- `JND = (x75 - x25) / 2`, in the same units as the stimulus values.
- Review `fit_warning`, `qc_warnings`, and `combined_flagged_trials.csv` before interpreting results.
- Rows excluded from fitting are preserved for audit; no raw data files are modified.